In [1]:
# Imports
import os
import dotenv
from typing import TypedDict
import datetime
import json
import logging  # added logging

# Web Scraping
import requests

# 3rd Party APIs
import finnhub

# Image processing
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image

# Audio
from pydub import AudioSegment
from pydub.playback import play

# LLM APIs
from openai import OpenAI

# User interface
import gradio as gr

In [2]:
# Config

# Load environment variables
dotenv.load_dotenv()

# Logging configuration (idempotent & de-duplicated)
def _init_logger(name: str = "finance_chat") -> logging.Logger:
    logger = logging.getLogger(name)

    # Remove any duplicate finance_chat handlers (those we previously added)
    existing_fc_handlers = [h for h in logger.handlers if getattr(h, "_finance_chat", False)]
    if len(existing_fc_handlers) > 1:
        for h in existing_fc_handlers[1:]:
            logger.removeHandler(h)

    # Remove any non-tagged handlers to prevent duplicate logs in notebooks
    for h in list(logger.handlers):
        if not getattr(h, "_finance_chat", False):
            logger.removeHandler(h)

    level_name = os.getenv("FINANCE_CHAT_LOG_LEVEL", "INFO").upper()
    level = getattr(logging, level_name, logging.INFO)

    # If a tagged handler remains, just ensure level/propagate and return
    if any(getattr(h, "_finance_chat", False) for h in logger.handlers):
        logger.setLevel(level)
        logger.propagate = False
        return logger

    # Otherwise create and attach a single tagged handler
    handler = logging.StreamHandler()
    handler._finance_chat = True  # tag to recognize later
    formatter = logging.Formatter('[%(asctime)s] %(levelname)s %(name)s - %(message)s')
    handler.setFormatter(formatter)

    logger.addHandler(handler)
    logger.setLevel(level)
    logger.propagate = False
    return logger

logger = _init_logger()

# Finnhub Client
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY")
finnhub_client = finnhub.Client(FINNHUB_API_KEY)

# LLM Client
openai = OpenAI()
SONAR_URL = "https://api.perplexity.ai/chat/completions"

# LLM models
OPEN_AI_MODEL = "gpt-5-nano"
OPEN_AI_IMAGE_MODEL = "dall-e-3"
OPEN_AI_AUDIO_MODEL = "tts-1"
GOOGLE_MODEL = "gemini-1.5-flash-latest"
SONAR_MODEL = "sonar-pro"

## Image Parameters
IMAGE_SIZE = "1024x1024"
N = 1
RESPONSE_FORMAT = "b64_json"
VOICE = "alloy"
AUDIO_FORMAT = "mp3"

# LLM Instructions
chat_system_message = """
You are a finance assistant that is able to summarize earnings call and report of the public companies and also
generate history charts of the selected finance metrics and desribe their trends.

Always be accurate. If you don't know the answer, say so.
"""

# Eearnings Call Agent
earnings_call_agent_system_message = """
You are a finance assistance that searches and summarizes the only last company earnings report and only from web search, 
for sales representative with focus on correlating facts with potential investments in AI/IT infrastructure returning the response 
in form of short statments/bullets. Ommit any side notes form search sources.

Be short and concise. If you cannot find a earnings call transcript
or report, say so.

Remove any links to the sources from the summary as this will be only text output in the textbox of the chatbot.

The example of the summary:
Company: Radiant Global Logistics Inc
* Very strong financial performance with 20% revenue growth year-over-year.
* Increased R&D expenses by 15%
* Increased CAPEX by 10%

Suggest to discuss where extra CAPEX and/or R&D investments can be made to improve the company's AI/IT infrastructure.

"""

earnings_call_agent_user_message = """
Summarize the most recent earnings call for {company_name} in year of {year}. 
"""


# Finance Metrics Agent
finance_metric_agent_system_message = """
You are a financial data assistant. Given a company name, a specific financial metric (e.g., revenue, net income, R&D, CAPEX), 
and a time window in years, extract a time series for that metric for the requested number of most recent years.

Always extract the data from finhub API using finhub_client.financials_reported function. Always call get_finance_metric function to get the data.
Parse the json response from finhub API to extract the relevant metric values for each fiscal quarter or year within the specified time window.

Return the result as a JSON object with the following structure:
{
  "x": ["FY21Q4", "FY22Q1", ..., "FY25Q4"],  // x-axis labels, each as fiscal year ex. FY22 or year-quarter ex. FY22Q1
  "y": [123.4, 150.2, ..., 210.0],           // y-axis values, one per x label rounded to 1 decimal place
  "x_label": "Fiscal Year & Quarter",        // Fiscal year if frquency is yearly or Fiscal Year & Quarter if frequency is quarterly
  "y_label": "Revenue (USD millions)"        // y_label should include the metric and its unit in parenthesis.
  "title": "Microsoft Corporation" // title should include the company name
}

Example for metric "revenue" for 4 years:
{
  "x": [ "FY22Q1", "FY22Q2", "FY22Q3", "FY22Q4", "FY23Q1", "FY23Q2", "FY23Q3", "FY23Q4", "FY24Q1", "FY24Q2", "FY24Q3", "FY24Q4", "FY25Q1", "FY25Q2", "FY25Q3", "FY25Q4"],
  "y": [120.5, 130.2, 128.7, 135.0, 140.1, 145.3, 150.2, 155.0, 160.4, 165.0, 170.2, 175.1, 180.0, 185.5, 190.2, 200.0],
  "x_label": "Fiscal Year & Quarter",
  "y_label": "Revenue (USD millions)" 
  "title": "Microsoft Corporation"
}

Reference below finhub API json response schema to extract:
The year, querter can be extracted from Filing secion using properties year and quarter.
The values can be extracted from FinancialLineItem using value property.
The metric can be inferred from FinancialLineItem concept or label property.
The unit can be extracted from unit property ofthe FinancialLineItem.

Json schema:
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "$id": "https://example.com/schemas/filing-report.schema.json",
  "title": "SEC Filing Report Collection",
  "type": "object",
  "additionalProperties": false,
  "required": ["cik", "data"],
  "properties": {
    "cik": {
      "type": "string",
      "description": "Company CIK as a string of up to 10 digits.",
      "pattern": "^\\d{1,10}$"
    },
    "data": {
      "type": "array",
      "minItems": 1,
      "items": { "$ref": "#/$defs/Filing" }
    }
  },
  "$defs": {
    "Filing": {
      "type": "object",
      "additionalProperties": false,
      "required": [
        "accessNumber",
        "symbol",
        "cik",
        "year",
        "quarter",
        "form",
        "startDate",
        "endDate",
        "filedDate",
        "acceptedDate",
        "report"
      ],
      "properties": {
        "accessNumber": {
          "type": "string",
          "description": "SEC accession number (##########-##-######).",
          "pattern": "^\\d{10}-\\d{2}-\\d{6}$"
        },
        "symbol": {
          "type": "string",
          "description": "Ticker symbol.",
          "pattern": "^[A-Z.-]{1,10}$"
        },
        "cik": {
          "type": "string",
          "description": "Company CIK as a string of up to 10 digits.",
          "pattern": "^\\d{1,10}$"
        },
        "year": {
          "type": "integer",
          "minimum": 1900,
          "maximum": 3000
        },
        "quarter": {
          "type": "integer",
          "minimum": 1,
          "maximum": 4
        },
        "form": {
          "type": "string",
          "description": "SEC form code (e.g., 10-Q, 10-K).",
          "pattern": "^[A-Z0-9-]+$"
        },
        "startDate": {
          "type": "string",
          "description": "Period start (YYYY-MM-DD HH:MM:SS).",
          "pattern": "^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}$"
        },
        "endDate": {
          "type": "string",
          "description": "Period end (YYYY-MM-DD HH:MM:SS).",
          "pattern": "^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}$"
        },
        "filedDate": {
          "type": "string",
          "description": "Date filed (YYYY-MM-DD HH:MM:SS).",
          "pattern": "^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}$"
        },
        "acceptedDate": {
          "type": "string",
          "description": "Date accepted (YYYY-MM-DD HH:MM:SS).",
          "pattern": "^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}$"
        },
        "report": { "$ref": "#/$defs/FilingReport" }
      }
    },
    "FilingReport": {
      "type": "object",
      "additionalProperties": false,
      "required": ["bs", "ic", "cf"],
      "properties": {
        "bs": {
          "type": "array",
          "description": "Balance sheet line items.",
          "minItems": 1,
          "items": { "$ref": "#/$defs/FinancialLineItem" }
        },
        "ic": {
          "type": "array",
          "description": "Income statement (P&L) line items.",
          "minItems": 1,
          "items": { "$ref": "#/$defs/FinancialLineItem" }
        },
        "cf": {
          "type": "array",
          "description": "Cash flow statement line items.",
          "minItems": 1,
          "items": { "$ref": "#/$defs/FinancialLineItem" }
        }
      }
    },
    "FinancialLineItem": {
      "type": "object",
      "additionalProperties": false,
      "required": ["concept", "unit", "label", "value"],
      "properties": {
        "concept": {
          "type": "string",
          "description": "XBRL concept name (e.g., us-gaap_..., msft_...).",
          "pattern": "^[a-z][a-z0-9-]*_[A-Za-z0-9]+$"
        },
        "unit": {
          "type": "string",
          "description": "Unit identifier (e.g., u_usd, u_shares).",
          "pattern": "^u_[a-z0-9]+$"
        },
        "label": {
          "type": "string"
        },
        "value": {
          "type": "number"
        }
      }
    }
  }
}


If a value is missing, use null in the y array. Always organize the x and y arrays chronologically from oldest to most recent.
Return only valid JSON as described above.
"""
finance_metric_agent_user_message = """
Extract the time series for the company {company_name} for the metric {metric_name} on {frequency} basis and {time_window_in_years} years.
Return results  as a JSON object describe in system message. Avoid any extra text, explanations, or comments.
Return only valid JSON. Do not include any other text.
"""

# Tools

# Test variables
company_name = "Microsoft"
metric_name = "revenue"
time_window_in_years = 5
frequency = 'annual'  # 'annual' or 'quarterly'

In [3]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Sequence, Protocol, runtime_checkable, Any, Dict, List, Optional


# === LLM Client Strategy Interfaces ===
@runtime_checkable
class LLMClientStrategy(Protocol):
    """Strategy interface for different LLM providers."""

    def chat(self, model: str, messages: Sequence[Dict[str, str]]) -> Any: ...


@dataclass
class OpenAIChatClient:
    """Concrete strategy wrapping OpenAI chat API."""

    client: Any

    def chat(self, model: str, messages: Sequence[Dict[str, str]]) -> Any:
        return self.client.chat.completions.create(model=model, messages=list(messages))


@dataclass
class PerplexityChatClient:
    """Concrete strategy wrapping Perplexity (SONAR) HTTP API."""

    api_url: str
    api_key: str

    def chat(self, model: str, messages: Sequence[Dict[str, str]]) -> Any:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        payload = {"model": model, "messages": list(messages)}
        resp = requests.post(self.api_url, headers=headers, json=payload, timeout=60)
        resp.raise_for_status()
        return resp.json()


# === Template Method Base Agent ===
class BaseLLMAgent(ABC):
    """Template + Strategy based base agent.

    Responsibilities:
    - Defines the generate() template method.
    - Delegates provider differences to an injected strategy (LLMClientStrategy).
    - Child classes override hooks for system prompt, user messages, and parsing.
    """

    MODEL: str = SONAR_MODEL  # default

    def __init__(
        self,
        client_strategy: Optional[LLMClientStrategy] = None,
        model_name: Optional[str] = None,
    ):
        self.model_name = model_name or self.MODEL
        # Default strategy: Perplexity if SONAR model else OpenAI
        if client_strategy:
            self._client = client_strategy
        else:
            if self.model_name.startswith("sonar"):
                self._client = PerplexityChatClient(
                    api_url=SONAR_URL, api_key=os.getenv("SONAR_API_KEY", "")
                )
            else:
                self._client = OpenAIChatClient(client=openai)

    # --- Hooks ---
    @property
    @abstractmethod
    def system_prompt(self) -> str: ...

    @abstractmethod
    def build_user_messages(self, *args, **kwargs) -> List[Dict[str, str]]:
        """Return a list of user (and optionally assistant) messages excluding system."""
        ...

    def build_messages(self, *args, **kwargs) -> List[Dict[str, str]]:
        messages = [{"role": "system", "content": self.system_prompt}]
        messages.extend(self.build_user_messages(*args, **kwargs))
        return messages

    def parse_response(self, raw: Any) -> str:
        """Default parser accommodating both OpenAI + Perplexity schemas."""
        # OpenAI returns object with .choices[0].message.content; Perplexity returns JSON
        try:
            # Attempt OpenAI style
            return raw.choices[0].message.content  # type: ignore[attr-defined]
        except Exception:
            # Attempt Perplexity style dict
            if isinstance(raw, dict):
                return raw.get("choices", [{}])[0].get("message", {}).get("content", "")
            return str(raw)

    # --- Template Method ---
    def generate(self, *args, **kwargs) -> str:
        messages = self.build_messages(*args, **kwargs)
        raw = self._client.chat(self.model_name, messages)
        return self.parse_response(raw)


# === Concrete Agents ===
class EarningsCallAgent(BaseLLMAgent):
    MODEL = SONAR_MODEL

    @property
    def system_prompt(self) -> str:
        return earnings_call_agent_system_message

    def build_user_messages(self, company_name: str) -> List[Dict[str, str]]:
        prompt = earnings_call_agent_user_message.format(
            company_name=company_name, year=datetime.datetime.now().year
        )
        return [{"role": "user", "content": prompt}]

    # Convenience wrapper
    def summarize_earnings_call(self, company_name: str) -> str:
        return self.generate(company_name)


class FinanceMetricsAgent(BaseLLMAgent):
    MODEL = OPEN_AI_MODEL

    @property
    def system_prompt(self) -> str:
        return finance_metric_agent_system_message

    def build_user_messages(
        self,
        company_name: str,
        metric_name: str,
        frequency: str,
        time_window_in_years: int,
        raw_data: Optional[Dict[str, Any]] = None,
    ) -> List[Dict[str, str]]:
        base = finance_metric_agent_user_message.format(
            company_name=company_name,
            metric_name=metric_name,
            frequency=frequency,
            time_window_in_years=time_window_in_years,
        )
        msgs = [{"role": "user", "content": base}]
        if raw_data:
            data_json = json.dumps(raw_data, default=str)
            msgs.append({"role": "user", "content": data_json})
        return msgs

    def summarize_financial_metrics(
        self,
        company_name: str,
        stock_symbol: str,
        metric_name: str,
        frequency: str = "annual",
        time_window_in_years: int = 5,
    ) -> str:
        raw = self._get_finance_metric(company_name, stock_symbol, frequency)
        summary = self.generate(
            company_name, metric_name, frequency, time_window_in_years, raw
        )
        try:
            logger.info(
                f"agent.metrics_summary company={company_name} symbol={stock_symbol} metric={metric_name} freq={frequency} years={time_window_in_years} -> {summary}"
            )
        except Exception as e:
            logger.warning(f"agent.metrics_summary.log_error: {e}")
        # Quality at the source: validate/repair JSON structure (x, y, x_label, y_label)
        fixed = self._coerce_metrics_json(
            summary, company_name, metric_name, frequency, time_window_in_years
        )
        return fixed

    def _is_valid_metrics_payload(self, obj: Any) -> bool:
        if not isinstance(obj, dict):
            return False
        x = obj.get("x")
        y = obj.get("y")
        xl = obj.get("x_label")
        yl = obj.get("y_label")
        title = obj.get("title")
        if (
            not isinstance(x, list)
            or not isinstance(y, list)
            or not isinstance(xl, str)
            or not isinstance(yl, str)
            or not isinstance(title, str)
        ):
            return False
        if len(x) != len(y) or len(x) == 0:
            return False
        # Ensure y contains numeric-like values
        try:
            [float(v) for v in y]
        except Exception:
            return False
        return True

    def _coerce_metrics_json(
        self,
        text: str,
        company_name: str,
        metric_name: str,
        frequency: str,
        time_window_in_years: int,
    ) -> str:
        # First, try strict JSON parsing
        def _strip_fences(t: str) -> str:
            t = t.strip()
            if t.startswith("```") and t.endswith("```"):
                t = t.strip("`")
                # crude fence removal fallback
            return t

        candidate = _strip_fences(text)
        try:
            obj = json.loads(candidate)
            if self._is_valid_metrics_payload(obj):
                return json.dumps(obj)
        except Exception:
            pass

        # Ask the LLM to repair: best-practice JSON repair prompt
        repair_msgs = [
            {
                "role": "system",
                "content": "You are a strict JSON reformatter. Return only valid JSON with keys x (list of strings), y (list of numbers), x_label (string), y_label (string), title (string that contains the company name). No extra text.",
            },
            {
                "role": "user",
                "content": f"Given the company {company_name}, metric {metric_name}, frequency {frequency}, time window {time_window_in_years} years, reformat the following into the required JSON schema.Content:{text}",
            },
        ]
        try:
            raw = self._client.chat(self.model_name, repair_msgs)
            fixed_text = self.parse_response(raw)
            obj = json.loads(fixed_text)
            if self._is_valid_metrics_payload(obj):
                return json.dumps(obj)
        except Exception as e:
            logger.warning(f"metrics_json.repair_failed: {e}")

        # As a final fallback, return original text so UI still shows a response
        return text

    def _get_finance_metric(
        self, company_name: str, stock_symbol: str, frequency: str = "annual"
    ) -> Dict[str, Any]:
        """Thin wrapper over finnhub financials_reported.

        Parameters
        ----------
        company_name : str
            Company name (currently unused but kept for future enrichment / logging).
        stock_symbol : str
            Ticker symbol accepted by Finnhub.
        frequency : str
            'annual' or 'quarterly'.
        """
        if frequency not in {"annual", "quarterly"}:
            raise ValueError("frequency must be 'annual' or 'quarterly'")
        financials_json = finnhub_client.financials_reported(symbol=stock_symbol, freq=frequency)
        # save datato json file for debugging
        with open(f"{company_name}_{stock_symbol}_financials_{frequency}.json", "w") as f:
            json.dump(financials_json, f, indent=2, default=str)
        return financials_json

In [4]:

# === Image Generation ===
class ImageGenerator:
    """Utility to produce matplotlib charts as PIL.Images from JSON-like time series."""

    # Static defaults (can be overridden in __init__)
    FIGSIZE = (7, 4)
    LINE_COLOR = "#1f77b4"
    LINE_WIDTH = 2
    MARKER = "o"
    GRID_ALPHA = 0.3
    TITLE_FMT = "{y_label} Trend"
    XLABEL = "Period"

    def __init__(self, figsize=None, line_color=None, line_width=None, marker=None, grid_alpha=None, title_fmt=None, xlabel=None):
        self.figsize = figsize or self.FIGSIZE
        self.line_color = line_color or self.LINE_COLOR
        self.line_width = line_width or self.LINE_WIDTH
        self.marker = marker or self.MARKER
        self.grid_alpha = grid_alpha or self.GRID_ALPHA
        self.title_fmt = title_fmt or self.TITLE_FMT
        self.xlabel = xlabel or self.XLABEL

    def create_matlotlib_chart(self, summary_json_text: str, company_name: Optional[str] = None) -> Image:
        """Create a trend chart from standardized JSON and return a PIL.Image via in-memory buffer.
        Expects JSON: { x: list[str], y: list[number], x_label: str, y_label: str, title: str }.
        The title is taken from 'title' if present, otherwise falls back to company_name.
        """
        # Strip code fences if present
        t = (summary_json_text or "").strip()
        if t.startswith('```') and t.endswith('```'):
            t = t.strip('`')
        obj = json.loads(t)
        if not isinstance(obj, dict):
            raise ValueError("Metrics payload must be a JSON object.")
        x = obj.get('x'); y = obj.get('y'); xl = obj.get('x_label'); yl = obj.get('y_label'); ti = obj.get('title')
        if not isinstance(x, list) or not isinstance(y, list) or not isinstance(xl, str) or not isinstance(yl, str):
            raise ValueError("Metrics payload missing required keys or wrong types.")
        if len(x) != len(y) or len(x) == 0:
            raise ValueError("x and y must be non-empty lists of equal length.")
        y_vals = [float(v) for v in y]

        plt.figure(figsize=self.figsize)
        plt.plot(x, y_vals, marker=self.marker, linewidth=self.line_width, color=self.line_color)
        # Title (prefer JSON 'title', else provided company name)
        title_str = ti if isinstance(ti, str) and ti.strip() else (company_name or None)
        if title_str:
            plt.title(title_str)
        plt.xlabel(xl)
        plt.ylabel(yl)
        plt.grid(True, alpha=self.grid_alpha)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        buf = BytesIO(); plt.savefig(buf, format="png"); plt.close(); buf.seek(0)
        return Image.open(buf)

def fn_get_finance_metrics(company_name: str, stock_symbol: str, metric_name: str, frequency: str = "annual", time_window_in_years: int = 5):
    """Return (text_json, PIL.Image|None) for the requested metric.
    Uses FinanceMetricsAgent and ImageGenerator.
    """
    agent = FinanceMetricsAgent()
    text = agent.summarize_financial_metrics(company_name, stock_symbol, metric_name, frequency, time_window_in_years)
    img = None
    try:
        img = ImageGenerator().create_matlotlib_chart(text, company_name=company_name)
    except Exception as e:
        logger.warning(f"chart.generate failed: {e}")
    return text, img

def fn_get_finance_metric_chart(summary_json_text: str, company_name: Optional[str] = None) -> Image:
    """Thin wrapper over ImageGenerator for convenience."""
    return ImageGenerator().create_matlotlib_chart(summary_json_text, company_name=company_name)

def fn_get_earnings_call_summary(company_name: str):
    """Return (text_summary, None) for latest earnings call."""
    agent = EarningsCallAgent()
    return agent.summarize_earnings_call(company_name), None

# === Tool Specifications ===
fn_get_finance_metrics_spec = {
    "name": "fn_get_finance_metrics",
    "description": "Get structured financial metric time series for a public company using the FinanceMetricsAgent. Returns JSON time series for the requested company, metric, frequency and time window in years.",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {"type": "string", "description": "The name of the company to get financial metrics for."},
            "stock_symbol": {"type": "string", "description": "The stock symbol of the company, search on internet by company name if not provided."},
            "metric_name": {"type": "string", "description": "The financial metric to extract (e.g., revenue, net income, research and development expenses)."},
            "frequency": {"type": "string", "enum": ["quarterly", "annual"], "default": "annual", "description": "Frequency of the financial report (quarterly or annual)."},
            "time_window_in_years": {"type": "integer", "default": 5, "minimum": 1, "description": "How many most recent years to include in the time series."},
        },
        "required": ["company_name", "stock_symbol", "metric_name"],
        "additionalProperties": False,
    },
}

fn_get_earnings_call_summary_spec = {
    "name": "fn_get_earnings_call_summary",
    "description": "Summarize the most recent earnings call for a public company using the EarningsCallAgent.",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {"type": "string", "description": "The name of the company to summarize the earnings call for."}
        },
        "required": ["company_name"],
        "additionalProperties": False,
    },
}

# Aggregate tool definitions for the OpenAI chat API
tools = [
    {"type": "function", "function": fn_get_finance_metrics_spec},
    {"type": "function", "function": fn_get_earnings_call_summary_spec},
]

In [5]:
import threading
from queue import Queue, Empty

class AudioPlayer:
    """Background TTS player that speaks small chunks sequentially.

    Best practice: decouple synthesis from UI thread and play in-order.
    """
    def __init__(self, max_queue:int=200):
        self.q: Queue[object] = Queue(maxsize=max_queue)
        self._thread = None
        self._stop = threading.Event()
        self._busy = threading.Event()  # True while playing a segment

    def start(self):
        if self._thread and self._thread.is_alive():
            return
        self._stop.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        try:
            self.q.put_nowait("")
        except Exception:
            pass

    def enqueue(self, text: str):
        """Backward compatible: enqueue text to synthesize and play."""
        self.enqueue_text(text)

    def enqueue_text(self, text: str):
        if not text:
            return
        try:
            self.q.put_nowait(("text", text))
        except Exception:
            pass

    # Global gate: disable server-side playback; use browser player instead
    ALLOW_SERVER_AUDIO = False
    def enqueue_audio(self, audio_segment: AudioSegment):
        if not self.ALLOW_SERVER_AUDIO:
            return
        try:
            self.q.put_nowait(("audio", audio_segment))
        except Exception:
            pass

    def _run(self):
        while not self._stop.is_set():
            try:
                item = self.q.get(timeout=0.25)
            except Empty:
                continue
            try:
                # Backward compatibility: plain str means text
                if isinstance(item, str):
                    kind, payload = "text", item
                else:
                    kind, payload = item
                if kind == "audio":
                    self._busy.set()
                    play(payload)
                    self._busy.clear()
                elif kind == "text":
                    if not payload:
                        continue
                    resp = openai.audio.speech.create(model=OPEN_AI_AUDIO_MODEL, voice=VOICE, input=payload)
                    audio_stream = BytesIO(resp.content)
                    audio = AudioSegment.from_file(audio_stream, format=AUDIO_FORMAT)
                    self._busy.set()
                    play(audio)
                    self._busy.clear()
            except Exception as e:
                logger.warning(f"tts.play_failed: {e}")

    def wait_idle(self, timeout: Optional[float] = None) -> bool:
        """Block until the current audio finishes (queue head).
        Returns True if idle, False on timeout.
        """
        if not self._busy.is_set():
            return True
        return not self._busy.wait(timeout=timeout)

# Singleton player
AUDIO_PLAYER = AudioPlayer()
AUDIO_PLAYER.start()

def talker(message: str):
    """Backwards-compatible: enqueue full message (non-streaming fallback)."""
    AUDIO_PLAYER.enqueue(message)

# Generate a TTS audio file for the full text (used after streaming completes)
import tempfile, os
def generate_tts_file(text: str) -> str:
    if not text or not text.strip():
        raise ValueError("No text for TTS")
    resp = openai.audio.speech.create(model=OPEN_AI_AUDIO_MODEL, voice=VOICE, input=text)
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=f'.{AUDIO_FORMAT}')
    tmp.write(resp.content)
    tmp.flush(); tmp.close()
    return tmp.name

import base64
def generate_tts_data_url(text: str) -> str:
    if not text or not text.strip():
        raise ValueError("No text for TTS")
    resp = openai.audio.speech.create(model=OPEN_AI_AUDIO_MODEL, voice=VOICE, input=text)
    b64 = base64.b64encode(resp.content).decode("ascii")
    mime = 'audio/mpeg' if AUDIO_FORMAT in ('mp3','mpeg') else f'audio/{AUDIO_FORMAT}'
    return f"data:{mime};base64,{b64}"

In [6]:
class ChatMessage(TypedDict):
    role: str
    content: str

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    name = tool_call.function.name
    logger.info(f"tool.dispatch name={name} args={arguments}")

    img = None
    if name == "fn_get_finance_metrics":
        result_text, img = fn_get_finance_metrics(
            arguments.get("company_name"),
            arguments.get("stock_symbol"),
            arguments.get("metric_name"),
            arguments.get("frequency", "annual"),
            arguments.get("time_window_in_years", 5)
        )
    elif name == "fn_get_earnings_call_summary":
        result_text, img = fn_get_earnings_call_summary(arguments.get("company_name"))
    else:
        logger.warning(f"tool.unknown name={name}")
        result_text = ""
    logger.info(f"tool.complete name={name} bytes={len(result_text) if isinstance(result_text,str) else 'n/a'}")
    tool_msg = {
        "role": "tool",
        "content": result_text,
        "tool_call_id": tool_call.id,
        "name": name,
    }
    return tool_msg, img

def chat(history: List[ChatMessage]):
    messages = [{"role": "system", "content": chat_system_message}] + history
    last_user = next((m['content'] for m in reversed(messages) if m['role']=='user'), None)
    logger.info(f"chat.start msg_count={len(messages)} last_user={last_user!r}")

    out_img = None
    # Decide whether tools are needed (non-streaming detection)
    first = openai.chat.completions.create(model=OPEN_AI_MODEL, messages=messages, tools=tools)
    choice = first.choices[0]
    if choice.finish_reason == "tool_calls":
        message = choice.message
        tc = message.tool_calls[0]
        logger.info(f"chat.tool_request name={tc.function.name}")
        tool_msg, img = handle_tool_call(message)
        if img is not None:
            out_img = img
        # Record assistant tool call and tool reply
        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [{
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }],
        })
        messages.append(tool_msg)
        logger.info(f"chat.tool_round_complete name={tc.function.name}")

    # Stream the final assistant reply (text only); show a small speaker icon bottom-right
    stream = openai.chat.completions.create(model=OPEN_AI_MODEL, messages=messages, stream=True)
    ui_history = [
        {"role": m.get("role"), "content": (m.get("content") or "")}
        for m in history if isinstance(m, dict) and m.get("role") in ("user", "assistant")
    ]
    ui_history.append({"role": "assistant", "content": ""})
    full = ""
    for chunk in stream:
        try:
            delta = chunk.choices[0].delta
            text = getattr(delta, 'content', None) or (delta.get('content') if isinstance(delta, dict) else None)
        except Exception:
            text = None
        if not text:
            continue
        full += text
        ui_history[-1]['content'] = full
        yield ui_history, (out_img if out_img is not None else gr.update())

    # After stream completes, embed clickable audio icon
    try:
        data_url = generate_tts_data_url(full)
        ui_history[-1]['content'] = full + f"""

<div style='text-align:right'><a class=\"play-audio\" href=\"{data_url}\" target=\"_blank\" title=\"Play audio\">🔊</a></div>"""
    except Exception as e:
        logger.warning(f"tts.inline_generate_failed: {e}")
    logger.info("chat.complete")
    yield ui_history, (out_img if out_img is not None else gr.update())


In [7]:
# # Example usage to demonstrate the chat function with the fn_get_finance_metrics tool
# user_message = f"Please provide me revenue report for {company_name} on {frequency} basis."
# history = [{"role": "user", "content": user_message}]
# history = chat(history)
# print(history)

In [8]:
# Example usage to demonstrate the chat function with the new fn_get_earnings_call_summary tool
# user_message = f"Please provide me earnings call summary for {company_name}."
# history = [{"role": "user", "content": user_message}]
# history = chat(history)
# print(history)

In [ ]:
# Gradio UI with chat + image + input
ALLOWED_ROLES = {"user", "assistant"}
def _sanitize_messages(msgs):
    """Clean chat history before sending to the LLM.

    - Only keep entries with role in ALLOWED_ROLES.
    - Strip any inline audio markup (audio tags, play links, enclosing divs) from assistant
      messages so that large base64 data URLs are not persisted in the history and sent back
      to the model on subsequent turns.  This preserves only the raw text of the assistant
      response, preventing token bloat.
    """
    import re
    cleaned = []
    for m in msgs:
        if not isinstance(m, dict):
            continue
        role = m.get("role"); content = m.get("content")
        if role in ALLOWED_ROLES and isinstance(content, (str, type(None))):
            text = content or ""
            # Remove <audio> tags
            text = re.sub(r"<audio[^>]*>.*?</audio>", "", text, flags=re.DOTALL)
            # Remove play-audio links
            text = re.sub(r"<a [^>]*play-audio[^>]*>.*?</a>", "", text, flags=re.DOTALL)
            # Remove the containing div for audio controls
            text = re.sub(r"<div[^>]*text-align:right[^>]*>.*?</div>", "", text, flags=re.DOTALL)
            cleaned.append({"role": role, "content": text.strip()})
    return cleaned


# ---------------------------------------------------------------------------
# Audio caching: create a temporary directory and SQLite database for per-message
# text-to-speech metadata.  Audio files are stored on disk and referenced by a
# unique ID.  When the Gradio server shuts down, the temporary directory and
# database are cleaned up automatically.
import sqlite3, tempfile, atexit, shutil, uuid

# Create a temp folder to hold audio files
AUDIO_TEMP_DIR = tempfile.mkdtemp(prefix="audio_cache_")
AUDIO_DB_PATH = os.path.join(AUDIO_TEMP_DIR, "audio_meta.db")
# Use a thread-safe connection for Gradio worker threads
AUDIO_DB_CONN = sqlite3.connect(AUDIO_DB_PATH, check_same_thread=False)
AUDIO_DB_LOCK = threading.Lock()
with AUDIO_DB_LOCK:
    _cursor = AUDIO_DB_CONN.cursor()
    _cursor.execute("CREATE TABLE IF NOT EXISTS audio_meta (id TEXT PRIMARY KEY, file_path TEXT)")
    AUDIO_DB_CONN.commit()

# In-memory mapping from chat index to audio id
audio_mapping: dict[int, str] = {}

def _cleanup_audio():
    try:
        with AUDIO_DB_LOCK:
            AUDIO_DB_CONN.close()
    except Exception:
        pass
    try:
        shutil.rmtree(AUDIO_TEMP_DIR, ignore_errors=True)
    except Exception:
        pass

atexit.register(_cleanup_audio)

# Override the chat function from earlier to incorporate audio caching.
def chat(history: List[ChatMessage]):
    messages = [{"role": "system", "content": chat_system_message}] + history
    last_user = next((m['content'] for m in reversed(messages) if m['role']=='user'), None)
    logger.info(f"chat.start msg_count={len(messages)} last_user={last_user!r}")
    out_img = None
    # Decide whether tools are needed (non-streaming detection)
    first = openai.chat.completions.create(model=OPEN_AI_MODEL, messages=messages, tools=tools)
    choice = first.choices[0]
    if choice.finish_reason == "tool_calls":
        message = choice.message
        tc = message.tool_calls[0]
        logger.info(f"chat.tool_request name={tc.function.name}")
        tool_msg, img = handle_tool_call(message)
        if img is not None:
            out_img = img
        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [{
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }],
        })
        messages.append(tool_msg)
        logger.info(f"chat.tool_round_complete name={tc.function.name}")
    # Stream the final assistant reply (text only)
    stream = openai.chat.completions.create(model=OPEN_AI_MODEL, messages=messages, stream=True)
    ui_history = [
        {"role": m.get("role"), "content": (m.get("content") or "")}
        for m in history if isinstance(m, dict) and m.get("role") in ("user", "assistant")
    ]
    ui_history.append({"role": "assistant", "content": ""})
    full = ""
    for chunk in stream:
        try:
            delta = chunk.choices[0].delta
            text = getattr(delta, 'content', None) or (delta.get('content') if isinstance(delta, dict) else None)
        except Exception:
            text = None
        if not text:
            continue
        full += text
        ui_history[-1]['content'] = full
        yield ui_history, (out_img if out_img is not None else gr.update())
    # After stream completes, generate a TTS file and attach a speaker icon
    try:
        audio_path_tmp = generate_tts_file(full)
        audio_id = str(uuid.uuid4())
        file_ext = os.path.splitext(audio_path_tmp)[1]
        final_path = os.path.join(AUDIO_TEMP_DIR, f"{audio_id}{file_ext}")
        try:
            shutil.move(audio_path_tmp, final_path)
        except Exception:
            shutil.copy(audio_path_tmp, final_path)
            os.remove(audio_path_tmp)
        try:
            with AUDIO_DB_LOCK:
                AUDIO_DB_CONN.execute(
                    "INSERT OR REPLACE INTO audio_meta (id, file_path) VALUES (?, ?)",
                    (audio_id, final_path),
                )
                AUDIO_DB_CONN.commit()
        except Exception as e:
            logger.warning(f"audio_meta.db.insert_failed: {e}")
        audio_index = len(ui_history) - 1
        audio_mapping[audio_index] = audio_id
        ui_history[-1]['content'] = full + f"\n\n<div style='text-align:right'><a class=\"play-audio\" data-audio-id=\"{audio_id}\">🔊</a></div>"

    except Exception as e:
        logger.warning(f"tts.file_generate_failed: {e}")
    logger.info("chat.complete")
    yield ui_history, (out_img if out_img is not None else gr.update())

with gr.Blocks() as ui:
    gr.HTML("""<style>
    .gr-chatbot .message, .gr-chatbot .wrap, .gr-chatbot .message-row { cursor: default !important; }
    a.play-audio { cursor: pointer !important; text-decoration: none; }
    </style>""")
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=400)
    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant")
    with gr.Row():
        clear = gr.Button("Clear")


    def do_entry(message, history):
        message = (message or "").strip()
        if not message:
            return "", history
        logger.info(f"UI received prompt: {message}")
        history = list(history or [])
        history += [{"role": "user", "content": message}]
        return "", _sanitize_messages(history)

    entry.submit(fn=do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
        fn=chat, inputs=chatbot, outputs=[chatbot, image_output]
    )

    def _do_clear():
        return []
    clear.click(_do_clear, None, [chatbot], queue=False)

    # Per-message play by clicking assistant bubble (uses event index)
    audio_output = gr.Audio(autoplay=True, visible=False)
    def _play_audio_on_select(evt: gr.SelectData):
        idx = evt.index
        audio_id = audio_mapping.get(idx)
        if not audio_id:
            return gr.update(value=None, visible=False)
        with AUDIO_DB_LOCK:
            row = AUDIO_DB_CONN.execute("SELECT file_path FROM audio_meta WHERE id=?", (audio_id,)).fetchone()
        if not row:
            return gr.update(value=None, visible=False)
        path = row[0]
        return gr.update(value=path, visible=True)
    chatbot.select(_play_audio_on_select, None, audio_output)

# Enable queuing so generator outputs stream reliably
ui.queue()
ui.launch(inbrowser=True, inline=False, share=False)


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


[2025-09-14 18:33:23,505] INFO finance_chat - UI received prompt: Give me last earning call of Microsoft.
[2025-09-14 18:33:23,676] INFO finance_chat - chat.start msg_count=2 last_user='Give me last earning call of Microsoft.'
[2025-09-14 18:33:26,014] INFO finance_chat - chat.tool_request name=fn_get_earnings_call_summary
[2025-09-14 18:33:26,015] INFO finance_chat - tool.dispatch name=fn_get_earnings_call_summary args={'company_name': 'Microsoft'}
[2025-09-14 18:33:32,103] INFO finance_chat - tool.complete name=fn_get_earnings_call_summary bytes=949
[2025-09-14 18:33:32,104] INFO finance_chat - chat.tool_round_complete name=fn_get_earnings_call_summary
[2025-09-14 18:33:54,773] INFO finance_chat - chat.complete
[2025-09-14 18:34:40,090] INFO finance_chat - UI received prompt: give me eps for last 10 years.
[2025-09-14 18:34:40,247] INFO finance_chat - chat.start msg_count=4 last_user='give me eps for last 10 years.'
[2025-09-14 18:34:43,288] INFO finance_chat - chat.tool_request name